# LAB03｜ADK + Groq + MCP：企業工具標準化
## V2.1 學員教學版｜Google ADK + Groq + Agent Skills + MCP + LINE

> **適合對象：** 不熟 Python、第一次接觸 AI Agent 的學員  
> **操作方式：** 由上到下依序按 ▶ 執行。  
> **重要原則：** 看到「👨‍🎓 學生修改區」才需要動手，其餘程式請先不要修改。


## 🎯 LAB03 學習目標

完成後你應該能回答：

- MCP 解決什麼問題？
- MCP Server 和 Agent 是什麼關係？
- Tool 為什麼不一定要寫死在 Agent 裡？
- 企業 ERP / CRM / MES 未來怎麼接進來？

### 本 LAB 架構

```text
Google ADK Agent
      │
      ▼
   McpToolset
      │
      ▼
   MCP Server
      │
   ┌──┼──────────────┐
   ▼  ▼      ▼       ▼
  客戶 產品   成本    產能
```


In [ ]:
# ============================================================
# STEP 1｜安裝本課需要的套件
# ============================================================
# 這一格的目的：
#   把 Google ADK、MCP、LiteLLM、pandas 安裝到 Colab。
#
# 為什麼需要？
#   Colab 預設沒有這些套件，因此第一次執行一定要安裝。
#
# 👨‍🎓 學生需要修改嗎？
#   不需要。直接按左側 ▶ 執行。
#
# ✅ 成功畫面：
#   執行結束後沒有紅色錯誤訊息即可。
#
# ❗常見錯誤：
#   1. 安裝很久：正常，第一次需要下載套件。
#   2. 出現版本衝突：請重新啟動執行階段後再執行一次。
# ============================================================

!pip -q install "google-adk[mcp]>=2.0.0,<3.0.0" "litellm>=1.84,<2" pandas
print("✅ 套件安裝完成")


In [ ]:
# ============================================================
# STEP 2｜上傳課程資料包
# ============================================================
# 這一格的目的：
#   把「企業模擬資料、Agent Skill、MCP Server」一次載入 Colab。
#
# 你要準備的檔案：
#   quotation_agent_v2_runtime.zip
#
# 👨‍🎓 學生修改區：
#   不用改程式，只要執行後選擇上述 ZIP 檔。
#
# ✅ 成功畫面：
#   看到「✅ 課程資料已載入」
#
# ❗常見錯誤：
#   - 上傳錯檔：重新執行這一格並選正確 ZIP。
#   - 沒看到 skills/ 或 data/：代表 ZIP 沒有成功解壓。
# ============================================================

from google.colab import files
import zipfile, os

if not os.path.exists("skills/quotation-decision/SKILL.md"):
    uploaded = files.upload()
    package = next((n for n in uploaded if n.endswith(".zip")), None)
    if not package:
        raise RuntimeError("❌ 請上傳 quotation_agent_v2_runtime.zip")
    with zipfile.ZipFile(package, "r") as z:
        z.extractall(".")

print("✅ 課程資料已載入")
print("目前重要資料夾：")
print(" - data/")
print(" - skills/quotation-decision/")
print(" - mcp_server/")


In [ ]:
# ============================================================
# STEP 3｜從 Colab Secrets 讀取 Groq API Key
# ============================================================
# 這一格的目的：
#   從 Google Colab 左側的「🔑 Secrets」安全讀取 GROQ_API_KEY。
#
# 👨‍🎓 學生事前準備：
#   1. 打開 Colab 左側選單
#   2. 點「🔑 Secrets」
#   3. 新增名稱：GROQ_API_KEY
#   4. Value 貼上你的 Groq API Key
#   5. 開啟「Notebook access」授權
#
# ⚠️ 不要把 API Key 直接寫在 Notebook 程式碼中。
#
# ✅ 成功畫面：
#   顯示「✅ 已從 Colab Secrets 讀取 GROQ_API_KEY」
#
# ❗常見錯誤：
#   KeyError / Secret not found
#   → 代表 Secrets 裡沒有 GROQ_API_KEY，或沒有開 Notebook access。
# ============================================================

import os
from google.colab import userdata

try:
    GROQ_API_KEY = userdata.get("GROQ_API_KEY")

    if not GROQ_API_KEY:
        raise ValueError("GROQ_API_KEY 為空值")

    os.environ["GROQ_API_KEY"] = GROQ_API_KEY

    # 課堂穩定模型
    GROQ_MODEL = "groq/qwen/qwen3.8-27b"

    print("✅ 已從 Colab Secrets 讀取 GROQ_API_KEY")
    print("✅ Groq 設定完成")
    print("Model =", GROQ_MODEL)
    print("教學穩定模式：reasoning_effort = none")

except Exception as e:
    raise RuntimeError(
        "❌ 無法讀取 Colab Secret：GROQ_API_KEY\n\n"
        "請到 Colab 左側「🔑 Secrets」新增：\n"
        "名稱：GROQ_API_KEY\n"
        "值：你的 Groq API Key\n"
        "並開啟 Notebook access。\n\n"
        f"原始錯誤：{e}"
    )


## STEP 4｜先看 MCP Server 裡有哪些 Tools

MCP Server 就像一間「企業能力服務中心」。

Agent 不需要自己知道資料在哪，只要知道：

> 我可以透過 MCP 找工具。


In [ ]:
# ============================================================
# STEP 4｜查看 MCP Server 程式
# ============================================================
#
# 你會看到這些 MCP Tools：
#
# customer_lookup
# product_lookup
# latest_cost
# order_history
# capacity_check
#
# 👨‍🎓 學生需要修改嗎？
#   不需要。只要看 Tool 名稱與用途。
# ============================================================

from pathlib import Path

server_text = Path(
    "mcp_server/quotation_mcp_server.py"
).read_text(encoding="utf-8")

print(server_text[:5000])


## STEP 5｜建立 `McpToolset`

白話意思：

> 「ADK Agent，企業 Tools 不在你身上，請去這個 MCP Server 找。」


In [ ]:
# ============================================================
# STEP 5｜連接 MCP Server
# ============================================================
#
# McpToolset：
#   讓 Google ADK Agent 可以使用 MCP Server 裡的 Tools。
#
# StdioConnectionParams：
#   在這堂課中，MCP Server 跑在同一個 Colab 環境。
#
# 正式企業環境未來可改成：
#   MCP Server → ERP / CRM / MES / APS
#
# 👨‍🎓 學生需要修改嗎？
#   不需要。
# ============================================================

import sys

from google.adk.tools.mcp_tool import McpToolset
from google.adk.tools.mcp_tool.mcp_session_manager import StdioConnectionParams
from mcp import StdioServerParameters

quotation_mcp = McpToolset(
    connection_params=StdioConnectionParams(
        server_params=StdioServerParameters(
            command=sys.executable,
            args=["mcp_server/quotation_mcp_server.py"],
        )
    )
)

print("✅ MCP Toolset 建立完成")


## STEP 6｜建立 MCP Agent

這次 Agent **不直接持有企業 Python functions**。

它只知道：

> 需要企業資料時，去問 MCP。


In [ ]:
# ============================================================
# STEP 6｜建立 MCP Agent
# ============================================================
#
# Model      → Groq
# Framework  → Google ADK
# Tools      → MCP
#
# 👨‍🎓 學生需要修改嗎？
#   不需要。
# ============================================================

from google.adk.agents import LlmAgent
from google.adk.models.lite_llm import LiteLlm

mcp_agent = LlmAgent(
    name="quotation_mcp_agent",

    model=LiteLlm(
        model=GROQ_MODEL,
        reasoning_effort="none",
    ),

    description="可以查詢企業報價資料的 Agent",

    instruction="""
你是製造業業務助理。

需要企業事實時，必須使用 MCP tools。

不可以自行猜測：
- 客戶信用
- 產品規格
- 成本
- 歷史價格
- 產能

請使用繁體中文回答。
""",

    tools=[
        quotation_mcp
    ],
)

print("✅ MCP Agent 建立完成")


In [ ]:
# ============================================================
# 共用工具｜讓我們可以直接「問 Agent 問題」
# ============================================================
# 這一格的目的：
#   Google ADK 的 Agent 需要 Runner + Session 才能真正執行。
#
# 你可以把它想成：
#
#   Agent = 員工
#   Runner = 讓員工開始工作的執行器
#   Session = 這一次對話的工作區
#
# 👨‍🎓 學生需要修改嗎？
#   不需要。
# ============================================================

from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types
import uuid

APP_NAME = "quotation_training"
USER_ID = "student"

async def ask_agent(agent, text: str):
    session_service = InMemorySessionService()
    session_id = str(uuid.uuid4())

    await session_service.create_session(
        app_name=APP_NAME,
        user_id=USER_ID,
        session_id=session_id,
    )

    runner = Runner(
        agent=agent,
        app_name=APP_NAME,
        session_service=session_service,
    )

    message = types.Content(
        role="user",
        parts=[types.Part(text=text)]
    )

    final_answer = "(沒有取得最終回答)"

    async for event in runner.run_async(
        user_id=USER_ID,
        session_id=session_id,
        new_message=message,
    ):
        if event.is_final_response() and event.content and event.content.parts:
            texts = [
                p.text for p in event.content.parts
                if getattr(p, "text", None)
            ]
            if texts:
                final_answer = "\n".join(texts)

    return final_answer

print("✅ Agent 執行工具已準備完成")


## STEP 7｜測試 MCP

👨‍🎓 **學生修改區**

你可以改成其他客戶或產品。


In [ ]:
# ============================================================
# 👨‍🎓 學生修改區
# ============================================================

QUESTION = """
ABC Electronics 的 CV420H
最新成本、最近成交價與 2026-10 可用產能是多少？
"""

# ============================================================
# ⚠️ 以下不需要修改
# ============================================================

print(await ask_agent(mcp_agent, QUESTION))


## ✅ LAB03 你已經完成

你現在已經知道：

- [x] MCP Server 可以提供多個 Tools
- [x] Agent 不必把所有企業功能寫死
- [x] ADK 可透過 `McpToolset` 使用 MCP
- [x] 課堂 CSV 未來可換成 ERP / CRM / MES

### 一句話記住

> **Skill = 怎麼做。**  
> **MCP = 去哪裡取得事實或執行能力。**

下一個 LAB：**把 ADK + Groq + Skill + MCP + LINE 全部串起來。**
